In [ ]:
import numpy as np
import pandas as pd

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def generate_synthetic_sigiriya(n=5000, seed=42):
    rng = np.random.default_rng(seed)

    # Time
    hour = rng.integers(0, 24, size=n)

    # API weather (reasonable ranges for Sri Lanka lowlands)
    temp_api  = rng.normal(29, 3.0, n).clip(20, 38)          # °C
    hum_api   = rng.normal(78, 10.0, n).clip(40, 100)        # %
    wind_api  = rng.normal(3.0, 1.8, n).clip(0, 12)          # m/s
    rain_api  = rng.gamma(shape=1.2, scale=2.0, size=n).clip(0, 40)  # mm/h-ish
    cloud_api = rng.normal(55, 25.0, n).clip(0, 100)         # %

    # Add "diurnal" effects (hotter mid-day, cooler early morning)
    diurnal = 2.5 * np.sin((hour - 6) / 24 * 2*np.pi)  # peak around ~12-14
    temp_api = (temp_api + diurnal).clip(20, 40)

    # Local sensor values: correlated with API + small bias/noise
    temp_local = (temp_api + rng.normal(0, 0.8, n) + rng.normal(0.2, 0.3, n)).clip(18, 42)
    hum_local  = (hum_api  + rng.normal(0, 3.0, n) + rng.normal(1.0, 1.0, n)).clip(30, 100)
    wind_local = (wind_api + rng.normal(0, 0.6, n)).clip(0, 15)

    # Local rain: sometimes higher than API due to microclimate; correlate with cloud + API rain
    rain_local = (0.7*rain_api + 0.03*cloud_api + rng.normal(0, 1.2, n)).clip(0, 60)

    # Visitor count: higher 8am-5pm, peak around 10-14
    base_vis = 40 + 120*np.exp(-((hour-12)/4.0)**2) + 60*np.exp(-((hour-10)/3.0)**2)
    visitor_count = (base_vis + rng.normal(0, 20, n)).clip(0, 350).round().astype(int)

    # --- Targets ---

    # Fog risk score (more likely early morning + high humidity/cloud + low wind + cooler temps)
    morning_bonus = np.where((hour <= 8) | (hour >= 19), 1.0, 0.0)
    fog_score = (
        0.08*(hum_local - 70) +
        0.03*(cloud_api - 50) +
        -0.25*(wind_local - 2.0) +
        -0.10*(temp_local - 28) +
        1.2*morning_bonus +
        rng.normal(0, 0.7, n)
    )
    fog_prob = sigmoid(fog_score)
    fog_risk = (rng.random(n) < fog_prob).astype(int)

    # Slip risk (wet + humid + cloudy + low wind)
    slip_score = (
        0.18*(rain_local - 1.5) +
        0.05*(hum_local - 75) +
        0.02*(cloud_api - 50) +
        -0.18*(wind_local - 2.0) +
        rng.normal(0, 0.8, n)
    )
    slip_prob = sigmoid(slip_score)
    slip_risk = (rng.random(n) < slip_prob).astype(int)

    # Heat stress (high temp + high humidity + crowding + mid-day)
    midday_bonus = np.where((hour >= 10) & (hour <= 15), 1.0, 0.0)
    heat_score = (
        0.20*(temp_local - 30) +
        0.06*(hum_local - 70) +
        0.004*(visitor_count - 80) +
        0.8*midday_bonus +
        -0.05*(wind_local - 2.0) +
        rng.normal(0, 0.9, n)
    )
    heat_prob = sigmoid(heat_score)
    heat_stress = (rng.random(n) < heat_prob).astype(int)

    df = pd.DataFrame({
        "temp_api": temp_api.round(2),
        "hum_api": hum_api.round(2),
        "wind_api": wind_api.round(2),
        "rain_api": rain_api.round(2),
        "cloud_api": cloud_api.round(2),
        "temp_local": temp_local.round(2),
        "hum_local": hum_local.round(2),
        "wind_local": wind_local.round(2),
        "rain_local": rain_local.round(2),
        "visitor_count": visitor_count,
        "hour": hour,
        "fog_risk": fog_risk,
        "slip_risk": slip_risk,
        "heat_stress": heat_stress
    })

    return df

if __name__ == "__main__":
    df = generate_synthetic_sigiriya(n=8000, seed=7)
    df.to_csv("sigiriya_synthetic_microclimate.csv", index=False)
    print(df.head())
    print("\nClass balance:")
    print(df[["fog_risk","slip_risk","heat_stress"]].mean())


   temp_api  hum_api  wind_api  rain_api  cloud_api  temp_local  hum_local  \
0     23.92    69.62      5.17      1.06      59.05       24.14      73.07   
1     25.36    88.34      0.00      7.93      40.71       24.66      88.90   
2     31.47    86.62      2.69      0.81      41.47       32.38      88.08   
3     23.81    86.99      5.29      4.08      32.26       23.82      92.24   
4     30.10    57.78      1.43      0.20      74.55       28.92      62.02   

   wind_local  rain_local  visitor_count  hour  fog_risk  slip_risk  \
0        4.80        1.45             36    22         0          0   
1        0.00        7.13            117    15         1          1   
2        2.79        0.12             70    16         0          1   
3        5.47        2.93             17    21         0          0   
4        1.58        2.29            191    13         1          1   

   heat_stress  
0            0  
1            0  
2            0  
3            1  
4            1  

C

In [9]:
import numpy as np
import pandas as pd
def generate_synthetic_sigiriya_data(n=20000, seed=42):
    rng = np.random.default_rng(seed)

    hour = rng.integers(0, 24, size=n)

    # Daily temperature pattern (°C) + noise
    base_temp = 26 + 4*np.sin((hour - 13) / 24 * 2*np.pi)  # warmer in afternoon
    temp_api = base_temp + rng.normal(0, 1.2, size=n)

    # Humidity (%) inversely related to temp + extra noise
    hum_api = 78 - 10*np.sin((hour - 13) / 24 * 2*np.pi) + rng.normal(0, 5, size=n)
    hum_api = np.clip(hum_api, 40, 98)

    # Wind (m/s) slightly higher in afternoon
    wind_api = 1.5 + 1.2*np.sin((hour - 12) / 24 * 2*np.pi) + rng.normal(0, 0.6, size=n)
    wind_api = np.clip(wind_api, 0, 10)

    # Cloud (0..1)
    cloud_api = rng.beta(2, 2, size=n)
    # Rain (mm/hr) correlated with cloud + humidity
    rain_api = np.maximum(
        0,
        (cloud_api - 0.55) * 10 + (hum_api - 75) * 0.05 + rng.normal(0, 0.8, size=n)
    )
    rain_api = np.clip(rain_api, 0, 30)

    # Local sensors deviate from API (microclimate + sensor noise)
    temp_local = temp_api + rng.normal(0, 0.8, size=n) + 0.3*(cloud_api - 0.5)
    hum_local  = np.clip(hum_api + rng.normal(0, 3.0, size=n) + 2*(rain_api > 0), 35, 100)
    wind_local = np.clip(wind_api + rng.normal(0, 0.5, size=n) - 0.2*(cloud_api > 0.7), 0, 12)
    rain_local = np.clip(rain_api + rng.normal(0, 0.6, size=n) + 0.5*(cloud_api > 0.75), 0, 35)

    # Visitor count: peaks morning + late afternoon
    visitor_count = (
        40
        + 180*np.exp(-((hour - 9) / 3.2)**2)
        + 220*np.exp(-((hour - 16) / 3.0)**2)
        + rng.normal(0, 20, size=n)
    )
    visitor_count = np.clip(visitor_count, 0, 600).astype(int)

    # ---------
    # Labels
    # ---------

    # Fog risk (0/1): high humidity + low wind + high cloud; cooler temps increase risk
    fog_score = (
        0.06*(hum_local - 70)
        + 1.2*(cloud_api - 0.5)
        - 0.35*(wind_local - 1.2)
        - 0.05*(temp_local - 26)
        + rng.normal(0, 0.6, size=n)
    )
    fog_prob = 1 / (1 + np.exp(-fog_score))
    fog_risk = (fog_prob > 0.55).astype(int)

    # Slip risk (0/1): rain + humidity + cloud; low wind slightly worse (surface stays wet)
    slip_score = (
        0.22*rain_local
        + 0.03*(hum_local - 60)
        + 0.8*(cloud_api - 0.4)
        - 0.08*wind_local
        + rng.normal(0, 0.8, size=n)
    )
    slip_prob = 1 / (1 + np.exp(-slip_score))
    slip_risk = (slip_prob > 0.55).astype(int)

    # Heat stress (0/1/2): temp + humidity + crowding (proxy for effort/thermal load)
    heat_index_like = (
        temp_local
        + 0.04*(hum_local - 50)
        + 0.004*visitor_count
        - 0.25*wind_local
        + rng.normal(0, 0.8, size=n)
    )

    heat_stress = np.zeros(n, dtype=int)
    heat_stress[heat_index_like >= 30.0] = 1
    heat_stress[heat_index_like >= 33.0] = 2

    df = pd.DataFrame({
        "temp_api": temp_api,
        "hum_api": hum_api,
        "wind_api": wind_api,
        "rain_api": rain_api,
        "cloud_api": cloud_api,
        "temp_local": temp_local,
        "hum_local": hum_local,
        "wind_local": wind_local,
        "rain_local": rain_local,
        "visitor_count": visitor_count,
        "hour": hour,
        "fog_risk": fog_risk,
        "slip_risk": slip_risk,
        "heat_stress": heat_stress
    })
    return df

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [3]:
def train_and_evaluate(df):
    feature_cols = [
        "temp_api","hum_api","wind_api","rain_api","cloud_api",
        "temp_local","hum_local","wind_local","rain_local",
        "visitor_count","hour"
    ]

    X = df[feature_cols]
    y_fog = df["fog_risk"]
    y_slip = df["slip_risk"]
    y_heat = df["heat_stress"]

    # Use the same split for all targets
    X_train, X_test, y_fog_train, y_fog_test = train_test_split(
        X, y_fog, test_size=0.2, random_state=42, stratify=y_fog
    )
    # Align other targets using indices
    y_slip_train = y_slip.loc[X_train.index]
    y_slip_test  = y_slip.loc[X_test.index]
    y_heat_train = y_heat.loc[X_train.index]
    y_heat_test  = y_heat.loc[X_test.index]

    # Baseline for fog: Logistic Regression (fast, explainable)
    fog_lr = LogisticRegression(max_iter=2000)
    fog_lr.fit(X_train, y_fog_train)

    # Strong baselines: Random Forests
    fog_rf = RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")
    slip_rf = RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced")
    heat_rf = RandomForestClassifier(n_estimators=300, random_state=42)

    fog_rf.fit(X_train, y_fog_train)
    slip_rf.fit(X_train, y_slip_train)
    heat_rf.fit(X_train, y_heat_train)

    # ---- Evaluate ----
    def eval_binary(name, model, y_true):
        pred = model.predict(X_test)
        acc = accuracy_score(y_true, pred)
        f1 = f1_score(y_true, pred)
        # AUC needs probabilities
        prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
        auc = roc_auc_score(y_true, prob) if prob is not None else None
        print(f"\n{name}")
        print(f"  Accuracy: {acc:.3f}")
        print(f"  F1:       {f1:.3f}")
        if auc is not None:
            print(f"  ROC-AUC:  {auc:.3f}")

    def eval_multiclass(name, model, y_true):
        pred = model.predict(X_test)
        acc = accuracy_score(y_true, pred)
        f1m = f1_score(y_true, pred, average="macro")
        print(f"\n{name}")
        print(f"  Accuracy:  {acc:.3f}")
        print(f"  Macro F1:   {f1m:.3f}")
        print("  Report:")
        print(classification_report(y_true, pred))

    eval_binary("Fog Risk - Logistic Regression", fog_lr, y_fog_test)
    eval_binary("Fog Risk - Random Forest", fog_rf, y_fog_test)
    eval_binary("Slip Risk - Random Forest", slip_rf, y_slip_test)
    eval_multiclass("Heat Stress - Random Forest", heat_rf, y_heat_test)

    return fog_rf, slip_rf, heat_rf

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("sigiriya_synthetic_microclimate.csv")

In [6]:
fog_model, slip_model, heat_model = train_and_evaluate(df)


Fog Risk - Logistic Regression
  Accuracy: 0.763
  F1:       0.841
  ROC-AUC:  0.789

Fog Risk - Random Forest
  Accuracy: 0.761
  F1:       0.840
  ROC-AUC:  0.775

Slip Risk - Random Forest
  Accuracy: 0.641
  F1:       0.707
  ROC-AUC:  0.684

Heat Stress - Random Forest
  Accuracy:  0.695
  Macro F1:   0.676
  Report:
              precision    recall  f1-score   support

           0       0.65      0.55      0.60       658
           1       0.72      0.80      0.75       942

    accuracy                           0.69      1600
   macro avg       0.69      0.67      0.68      1600
weighted avg       0.69      0.69      0.69      1600



In [10]:
df = generate_synthetic_sigiriya_data(n=20000, seed=42)
fog_model, slip_model, heat_model = train_and_evaluate(df)


Fog Risk - Logistic Regression
  Accuracy: 0.849
  F1:       0.867
  ROC-AUC:  0.931

Fog Risk - Random Forest
  Accuracy: 0.844
  F1:       0.864
  ROC-AUC:  0.923

Slip Risk - Random Forest
  Accuracy: 0.725
  F1:       0.821
  ROC-AUC:  0.748

Heat Stress - Random Forest
  Accuracy:  0.920
  Macro F1:   0.761
  Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96      3210
           1       0.77      0.79      0.78       728
           2       0.83      0.40      0.54        62

    accuracy                           0.92      4000
   macro avg       0.85      0.72      0.76      4000
weighted avg       0.92      0.92      0.92      4000



In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


import joblib

# -----------------------------
# Synthetic data generator
# -----------------------------


def make_synthetic_sigiriya_data(n=20000, seed=42):
    rng = np.random.default_rng(seed)
    hour = rng.integers(0, 24, size=n)

    # --- "API" weather (broader, noisier) ---
    temp_api = rng.normal(29, 3.5, n).clip(18, 38)
    hum_api = rng.normal(75, 12, n).clip(35, 100)
    wind_api = rng.normal(3.0, 1.6, n).clip(0, 12)
    cloud_api = rng.normal(55, 25, n).clip(0, 100)
    rain_api = (rng.gamma(shape=1.2, scale=2.0, size=n) * (cloud_api / 100)).clip(0, 30)

    # --- Local IoT sensors (close to API but with microclimate differences) ---
    # Sigiriya rock area can be slightly warmer, more humid, wind differs near rock faces
    temp_local = (temp_api + rng.normal(0.6, 0.8, n)).clip(18, 40)
    hum_local = (hum_api + rng.normal(2.0, 4.0, n)).clip(30, 100)
    wind_local = (wind_api + rng.normal(-0.2, 0.7, n)).clip(0, 14)
    rain_local = (rain_api + rng.normal(0.0, 1.5, n)).clip(0, 35)

    # Visitors: more in daytime; weekends-ish effect via random spikes
    base_visitors = np.where(
        (hour >= 7) & (hour <= 17), rng.normal(250, 120, n), rng.normal(60, 40, n)
    )

    spikes = rng.choice([0, 1], size=n, p=[0.9, 0.1]) * rng.normal(200, 80, n)
    visitor_count = (base_visitors + spikes).clip(0, 900).round().astype(int)

    # -----------------------------
    # Risk label rules (with noise)
    # -----------------------------

    # Fog proxy: high humidity + low wind + high cloud + small temp-dew spread
    # dew point approx (simple): Td ≈ T - (100-RH)/5

    dew_local = temp_local - (100 - hum_local) / 5.0
    temp_dew_spread = temp_local - dew_local

    fog_score = (
        0.9 * (hum_local / 100)
        + 0.6 * (cloud_api / 100)
        + 0.4 * (rain_local / 10)
        - 0.7 * (wind_local / 10)
        - 0.6 * (temp_dew_spread / 10)
    )

    # more fog risk early morning / night

    fog_score += np.where((hour <= 6) | (hour >= 19), 0.25, 0.0)
    fog_score += rng.normal(0, 0.15, n)
    fog_risk = (fog_score > 0.85).astype(int)

    # Slip proxy: rain + humidity + crowd + night
    slip_score = (
        0.9 * (rain_local / 10)
        + 0.3 * (hum_local / 100)
        - 0.2 * (wind_local / 10)
        + 0.25 * (visitor_count / 900)
    )

    slip_score += np.where(
        (hour <= 6) | (hour >= 18), 0.25, 0.0
    )  # slippery in low light
    slip_score += rng.normal(0, 0.18, n)

    slip_risk = (slip_score > 0.75).astype(int)

    # Heat stress proxy: heat index-ish + crowd load

    heat_index = temp_local + 0.08 * hum_local - 0.6 * wind_local

    heat_index += 2.0 * (visitor_count / 900)  # crowd heat / congestion effect

    heat_index += rng.normal(0, 0.8, n)

    # 3 classes: 0=low, 1=medium, 2=high

    heat_stress = np.select(
        [heat_index < 30, (heat_index >= 30) & (heat_index < 34), heat_index >= 34],
        [0, 1, 2],
    ).astype(int)

    df = pd.DataFrame(
        {
            "temp_api": temp_api,
            "hum_api": hum_api,
            "wind_api": wind_api,
            "rain_api": rain_api,
            "cloud_api": cloud_api,
            "temp_local": temp_local,
            "hum_local": hum_local,
            "wind_local": wind_local,
            "rain_local": rain_local,
            "visitor_count": visitor_count,
            "hour": hour,
            "fog_risk": fog_risk,
            "slip_risk": slip_risk,
            "heat_stress": heat_stress,
        }
    )

    return df


# -----------------------------


# Train + compare models


# -----------------------------


FEATURES = [
    "temp_api",
    "hum_api",
    "wind_api",
    "rain_api",
    "cloud_api",
    "temp_local",
    "hum_local",
    "wind_local",
    "rain_local",
    "visitor_count",
    "hour",
]


def evaluate_models_for_target(df, target, is_multiclass=False, seed=42):

    X = df[FEATURES]

    y = df[target]

    stratify = y if y.nunique() > 1 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=stratify
    )

    models = {
        "LogReg": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=500,
                        solver="lbfgs",
                        class_weight="balanced" if not is_multiclass else None,
                    ),
                ),
            ]
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            random_state=seed,
            class_weight="balanced" if not is_multiclass else None,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=seed),
    }

    results = []

    trained = {}

    for name, model in models.items():

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        acc = accuracy_score(y_test, preds)

        # F1: binary -> 'binary', multiclass -> 'macro'

        avg = "macro" if is_multiclass else "binary"

        f1 = f1_score(y_test, preds, average=avg)

        results.append((name, acc, f1))

        trained[name] = model

    results_df = pd.DataFrame(results, columns=["model", "accuracy", "f1"])

    best_name = results_df.sort_values(["f1", "accuracy"], ascending=False).iloc[0][
        "model"
    ]

    best_model = trained[best_name]

    # Print best model details

    print(f"\n=== Target: {target} | BEST: {best_name} ===")

    print(
        results_df.sort_values(["f1", "accuracy"], ascending=False).to_string(
            index=False
        )
    )

    best_preds = best_model.predict(X_test)

    print("\nClassification report (best):")

    print(classification_report(y_test, best_preds))

    print("Confusion matrix (best):")

    print(confusion_matrix(y_test, best_preds))

    return best_name, best_model, results_df


In [14]:
# 1) Make synthetic dataset
df = make_synthetic_sigiriya_data(n=20000, seed=42)
print(df.head())

# 2) Train/compare for each target
fog_best_name, fog_best_model, fog_results = evaluate_models_for_target(df, "fog_risk", is_multiclass=False)
slip_best_name, slip_best_model, slip_results = evaluate_models_for_target(df, "slip_risk", is_multiclass=False)
heat_best_name, heat_best_model, heat_results = evaluate_models_for_target(df, "heat_stress", is_multiclass=True)

# 3) Save best models (deploy later in your module)
joblib.dump(fog_best_model, "best_fog_risk_model.joblib")
joblib.dump(slip_best_model, "best_slip_risk_model.joblib")
joblib.dump(heat_best_model, "best_heat_stress_model.joblib")

print("\nSaved:")
print(" - best_fog_risk_model.joblib")
print(" - best_slip_risk_model.joblib")
print(" - best_heat_stress_model.joblib")

    temp_api    hum_api  wind_api  rain_api   cloud_api  temp_local  \
0  22.408351  67.520078  3.579124  2.307674  100.000000   23.394530   
1  32.956672  69.183576  7.210668  0.245613   49.737331   34.342095   
2  33.139537  58.591643  4.298363  2.220099   55.550619   33.893969   
3  27.339716  74.564740  1.529962  0.231102   40.932318   27.110883   
4  30.824466  71.780973  2.342717  0.414036   50.206065   31.740200   

   hum_local  wind_local  rain_local  visitor_count  hour  fog_risk  \
0  71.427749    4.951730    5.044928             88     2         1   
1  75.618068    6.881087    0.000000              0    18         0   
2  61.241734    4.421835    1.600484             43    15         0   
3  74.206347    1.322247    0.000000            153    10         0   
4  73.933001    2.394754    2.695170             10    10         0   

   slip_risk  heat_stress  
0          0            0  
1          0            2  
2          0            2  
3          0            1  
4     

In [16]:
df.to_csv("sigiriya_synthetic_microclimate.csv", index=False)

In [21]:
import glob
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

FEATURES = [
    "temp_api","hum_api","wind_api","rain_api","cloud_api",
    "temp_local","hum_local","wind_local","rain_local",
    "visitor_count","hour"
]
TARGETS = ["fog_risk", "slip_risk", "heat_stress"]

def load_best_models(model_dir="saved_models"):
    models = {}
    for t in TARGETS:
        candidates = glob.glob(f"{model_dir}/best_{t}*.joblib")
        if not candidates:
            raise FileNotFoundError(f"No saved model found for target={t} in {model_dir}")
        # if multiple, pick the latest by name sort (or you can use timestamp)
        path = sorted(candidates)[-1]
        models[t] = joblib.load(path)
    return models

def test_saved_models(df_test: pd.DataFrame, model_dir="saved_models"):
    models = load_best_models(model_dir)
    X = df_test[FEATURES]

    report = {}
    for t, model in models.items():
        y_true = df_test[t].astype(int)
        y_pred = model.predict(X)

        report[t] = {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro"))
        }

    return report

def predict_risks_live(row_df: pd.DataFrame, model_dir="saved_models"):
    """
    row_df: DataFrame with 1 row containing FEATURES columns.
    Returns: dict {target: predicted_class}
    """
    models = load_best_models(model_dir)
    X = row_df[FEATURES]
    return {t: int(models[t].predict(X)[0]) for t in TARGETS}




In [23]:
_ , df_test = df.iloc[:10000], df.iloc[10000:]

print(test_saved_models(df_test))

{'fog_risk': {'accuracy': 0.8901, 'macro_f1': 0.8820464007135345}, 'slip_risk': {'accuracy': 0.8889, 'macro_f1': 0.7471830077125567}, 'heat_stress': {'accuracy': 0.9039, 'macro_f1': 0.8864528151023858}}


In [24]:
live_pred = predict_risks_live(df_test.head(1))
print(live_pred)

{'fog_risk': 0, 'slip_risk': 0, 'heat_stress': 2}
